# [Day 3 세미나] 에이전틱 AI 기반 업무 자동화 시스템
## LangGraph와 MCP 프로토콜을 활용한 이메일 문서 정리 아키텍처

**발표일자:** 2026. 05. 29  
**핵심 기술:** LangGraph, Model Context Protocol(MCP), Webhook API

## 1. LangGraph(랭그래프)란 무엇인가?



### 🎬 "에이전트들의 협업 동선과 스케줄을 짜주는 무대 감독"

* **개념 정의:** AI 모델들이 일하는 순서와 흐름을 **그래프 구조(노드와 엣지)**로 설계하고 제어하는 오케스트레이션 프레임워크입니다.
* **도입 가치:** 일반적인 챗봇은 일회성 답변만 하지만, 실제 현업은 `[분석 ➔ 분류 ➔ 처리 ➔ 보고]`의 단계적 협업이 필요합니다. 
* **핵심 기능:** 공통 메모리인 `State` 객체를 기반으로, 에이전트 중 하나가 네트워크 에러 등으로 실패하더라도 데이터가 날아가지 않고 다음 단계로 안전하게 이어지도록 조율합니다.

## 2. MCP(Model Context Protocol)란 무엇인가?



### 🔌 "AI 에이전트와 외부 세계를 연결하는 표준 USB 허브"

* **개념 정의:** 앤트로픽(Anthropic)이 제안한 개방형 표준 프로토콜로, **AI 모델(Client)이 외부 데이터 소스나 도구(Server)에 통일된 방식으로 접근**할 수 있도록 규격화한 통신 규칙입니다.
* **도입 가치:** 기존에는 메일 API, 사내 ERP, 데이터베이스 등 새로운 기능을 붙일 때마다 연동 코드를 매번 새로 짜야 했습니다. 
* **혁신성:** MCP를 도입하면 연동 기능은 MCP 서버(`my_mcp_server.py`)가 전담하고, 에이전트 본체는 규격화된 포트에 꽂기만 하면 되므로 시스템 결합도가 획기적으로 낮아집니다.

## 3. 스마트 오피스 구현을 위한 핵심 메커니즘



### 🔄 이메일 수신 트리거 (RPA와의 결정적 차이)
* **기존 RPA 방식 (Polling):** 프로그램이 1분마다 이메일 서버에 "새 메일 왔어?"라고 계속 물어보는 방식 (지연 발생 및 서버 리소스 낭비).
* **에이전틱 AI 방식 (Webhook):** 메일이 서버에 도착하는 즉시 백엔드를 깨우는 **실시간 Push 알림** 구조입니다. 사람이 수동으로 실행 버튼을 누르지 않아도 **이메일 데이터 자체가 에이전트를 가동시키는 트리거** 역할을 합니다.

### 📋 초기 데이터 상태 객체 (`EmailState`) 구조
메일이 접수되는 순간, AI가 즉시 인지할 수 있도록 아래와 같은 전역 딕셔너리 구조로 변환되어 그래프에 주입됩니다.
```python
initial_state = {
    "mail_id": "msg_123456789",
    "sender": "partner@company.com",
    "subject": "[견적요청] 신형 조립 라인 비전 센서 단가 건",
    "body": "안녕하세요. 첨부한 엑셀 규격서 확인 후 내일까지 견적서 회신 부탁드립니다.",
    "attachments": ["specification.xlsx"],
    "agent_steps": ["ℹ️ 이메일 Webhook 수신 완료"]
}

---
## 5. 결론 및 엔지니어링 가치 (핵심 메일 집중 및 불필요 메일 필터링)

### 🚨 1) 임원(C-Level) 지시 사항 최우선 필터링 및 자율 대응
* **실무 핵심 응용:** 수백 건의 메일 중 임원진이 보낸 핵심 지시 사항이나 긴급 이슈 보고를 AI가 실시간으로 분석합니다.
* **우선순위 자율 할당:** 메일 본문에서 "다음 주까지 보고", "즉시 확인 바람" 등의 맥락을 파악하여, 관련 기술 매뉴얼(Vector DB)을 즉시 매핑하고 담당 엔지니어에게 **[임원 지시 - 긴급]** 알림을 발행합니다.

### 🗑️ 2) 관심사의 완벽한 분리 (Loose Coupling)를 통한 불필요 메일 격리
* **회의 일정 및 단순 알림 분리:** 단순 "회의 일정 안내", "시스템 정기 점검", "일반 공지" 등 당장 분석이 필요 없는 메일들을 메인 업무 파이프라인에서 완벽히 격리(Filter Out)합니다.
* **별도 보관함 이동:** 이러한 불필요 메일들은 `MailTriageAgent` 레이어에서 하위 업무 노드로 보내지 않고 **[회의/일반 알림 별도 보관함]**으로 바로 아카이브하여, 에이전트의 연산 리소스를 아끼고 핵심 업무의 데이터 오염을 방지합니다.

### 💡 3) 단순 업무 제거 및 리소스 고부가가치화
* 출근 직후 중요 메일을 찾아 헤매거나 단순 메일을 확인하느라 소모하던 리서치 공백을 AI에게 완전히 위임합니다. 
* 관리자와 엔지니어는 출근과 동시에 정제된 **'임원 지시 현황 및 동향 브리핑 문서'만 확인**하고 곧바로 설계와 의사결정에 집중할 수 있습니다.
   

## 6. 라이브 데모 실행 및 현장 센싱 시연 (Live Demo)

이론과 확장 사례로 검증한 에이전틱 AI 아키텍처 기반의 **'로봇 최신 AI 트렌드 수집기'**를 로컬 환경에서 직접 기동해 보겠습니다.

### 🏃‍♂️ 1) 백엔드 도구 서버 및 프론트엔드 UI 실행 과정
* **터미널 1 (MCP Tool Server 구동):** ```bash
  python src/my_mcp_server.py

$env:MY_MCP_URL="[http://127.0.0.1:8766/mcp](http://127.0.0.1:8766/mcp)"
streamlit run src/my_mcp_app.py


## 7. 데모로 확인한 로봇 AI 트렌드 센싱 플로차트 분석

방금 실행한 라이브 데모의 백엔드에서 **ArXiv 논문, 구글 뉴스, 로봇진, 레딧 데이터**가 어떻게 유기적으로 수집되고 결합되었는지 정밀 레이아웃으로 확인합니다.

```mermaid
graph TD
    A[사용자 자연어 입력 / Streamlit UI] --> B(CoordinatorAgent: 📁 my_mcp_graph.py)
    B -->|top_k 동적 보정 및 기획| C[RFMFetcherAgent]
    
    C -->|1. RFM 데이터 요청| G{MCP Client: 📁 my_mcp_client.py}
    C -->|rfm_results 누적| D[VLAFetcherAgent]
    
    D -->|2. VLA 데이터 요청| G
    D -->|vla_results 누적| E[ImitationFetcherAgent]
    
    E -->|3. 모방학습 데이터 요청| G
    E -->|imitation_results 누적| F[RedditFetcherAgent]
    
    F -->|4. 레딧 커뮤니티 요청| G
    F -->|reddit_results 누적| L[ReportAgent: 결과 종합 편집]
    
    G <-->|stdio / HTTP 프로토콜 통신| H[FastMCP Server: 📁 my_mcp_server.py]
    H --> I[(ArXiv API: 최신 논문)]
    H --> J[(Google News RSS / 로봇진 HTML)]
    H --> K[(Reddit r/robotics Feed)]
    
    L -->|공유 State 취합 및 포맷팅| M[outputs/robot_news_report.md 저장]

## 8. 한눈에 이해하는 에이전트 협업 방식: "시장 바구니 이어달리기"

에이전트들이 일하는 방식은 복잡한 프로그램 통신이 아니라, 하나의 바구니를 들고 순서대로 달리는 **이어달리기**와 똑같습니다.

### 🧺 기존 방식 vs 에이전틱 AI 방식

* **❌ 기존 방식 (메모리 공유 없음):** * 1번 주자가 2번 주자에게 데이터를 직접 손으로 쥐여줍니다. 
  * 만약 3번 주자(레딧)가 달리다가 넘어지면(에러 발생), 앞 사람이 준 데이터까지 길바닥에 전부 쏟아져서 처음부터 다시 달려야 합니다.

* **🎯 본 시스템 방식 (NewsState 공유 메모리):**
  * 주자들이 데이터를 직접 주고받지 않고, **공동으로 쓰는 '시장 바구니(NewsState)'** 하나를 바닥에 둡니다.
  * 순서대로 와서 자기가 수집한 물건만 바구니에 담고 다음 사람을 깨웁니다.

```mermaid
graph LR
    subgraph NewsState [🧺 공통 바구니: 전역 메모리]
        M1[RFM 논문]
        M2[VLA 기사]
        M3[모방학습 자료]
        M4[레딧 글]
    end

    A[1번 RFM 에이전트] -->|담기| M1
    A -->|깨우기| B[2번 VLA 에이전트]
    B -->|담기| M2
    B -->|깨우기| C[3번 모방학습 에이전트]
    C -->|담기| M3
    C -->|깨우기| D[4번 레딧 에이전트]
    D -->|담기| M4
    D -->|배달| E[최종 마크다운 보고서]

    style NewsState fill:#FFF2CC,stroke:#FFC000,stroke-width:2px;

## 9. 비유를 코드로 확인하기: "담기"와 "깨우기"의 실체

앞서 말씀드린 **'바구니에 담기'**와 **'다음 사람 깨우기'**는 LangGraph 코드 내에서 매우 직관적인 규칙으로 구현되어 있습니다.

### 🧺 1) 바구니에 "담기" (State Mutation)
각 에이전트 함수(Node)는 작업이 끝날 때, 자신이 수집한 데이터를 **딕셔너리 형태(`dict`)로 리턴**하기만 하면 됩니다. LangGraph가 이 리턴값을 전역 바구니(`NewsState`)에 자동으로 차곡차곡 업데이트(누적)해 줍니다.

```python
# src/my_mcp_graph.py 내부의 RFM 수집 노드 예시
def rfm_agent_node(state: NewsState) -> dict:
    # 1. MCP 서버를 호출하여 최신 RFM 논문을 수집
    raw_data = call_tool("search_rfm_news", {"query": state["user_query"]})
    
    # 2. 전역 바구니(NewsState)의 'rfm_results' 칸에 데이터 "담기"
    return {
        "rfm_results": raw_data,
        "agent_steps": ["✅ RFMFetcherAgent가 바구니에 논문 담기 완료"]
    }

## 11. `call_tool` 함수의 실체: "만능 리모컨 인터페이스"

`my_mcp_client.py`의 `call_tool`은 특정 수집 함수를 직접 가리키는 포인터가 아니라, 문자열 이름(`tool_name`)을 받아 해당하는 외부 도구를 동적으로 찾아서 실행해 주는 **만능 리모컨**입니다.

### 🔌 1) 왜 함수를 직접 호출하지 않고 `call_tool`을 거치는가? (완벽한 인프라 은닉)
만약 에이전트들이 `search_rfm_news()` 같은 함수를 직접 호출하게 짜면, 나중에 통신 방식(stdio 서브프로세스 방식 또는 HTTP 웹 서버 방식)이 바뀔 때 모든 에이전트 코드를 다 뜯어고쳐야 합니다.

`call_tool`은 그 복잡한 통신 방식과 비동기(Asyncio) 스레드 꼬임 문제를 내부에서 완벽히 방어해 주는 **인프라 방화벽** 역할을 합니다.

### 🔄 2) 코드 상에서의 동적 매핑 흐름

```mermaid
graph LR
    subgraph my_mcp_graph.py [🧠 상위 오케스트레이터]
        A[rfm_agent_node] -->|① 문자열로 도구 요청| B("call_tool('search_rfm_news', ...)")
    end

    subgraph my_mcp_client.py [🔌 중간 인터페이스 레이어]
        B -->|② 통신 방식 판단| C{HTTP / stdio}
    end

    subgraph my_mcp_server.py [🛠️ 하위 MCP 도구 서버]
        C -->|③ 도구 이름 매핑| D["@mcp.tool<br>def search_rfm_news()"]
        C -->|④ 도구 이름 매핑| E["@mcp.tool<br>def search_vla_news()"]
    end

    style my_mcp_client.py fill:#FFF2CC,stroke:#FFC000,stroke-width:2px;

## 12. `ThreadPoolExecutor`와 `asyncio.run` 코드의 진짜 의미

`my_mcp_client.py` 내부의 이 복잡한 코드는 데이터를 담는 로직이 아니라, **화면(Streamlit)과 AI 엔진(Asyncio)이 부딪혀서 프로그램이 다운되는 것을 막는 '방어막'**입니다.

```python
# 📁 src/my_mcp_client.py 내부의 실제 인프라 방어막 코드
if running_loop and running_loop.is_running():
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        future = pool.submit(asyncio.run, _call_tool_async(tool_name, tool_input))
        raw = future.result()

gantt
    title call_tool 4회 순차 실행 시 스레드 점유 상태
    dateFormat  X
    axisFormat %s초
    
    section 메인 스레드 (LangGraph)
    RFM 노드 가동           :active, a1, 0, 2
    VLA 노드 가동           :active, a2, 2, 4
    모방학습 노드 가동      :active, a3, 4, 6
    레딧 노드 가동          :active, a4, 6, 8
    
    section 격리 스레드 (ThreadPool)
    스레드 #1 생성 및 소멸 (RFM 수집)   :crit, t1, 0, 2
    스레드 #2 생성 및 소멸 (VLA 수집)   :crit, t2, 2, 4
    스레드 #3 생성 및 소멸 (모방학습 수집) :crit, t3, 4, 6
    스레드 #4 생성 및 소멸 (레딧 수집)   :crit, t4, 6, 8

### 🗣️ 핵심 구조

결론부터 말씀드리면, **동시에 떠 있는 스레드는 언제나 딱 1개**입니다. 

비결은 파이썬의 `with` 문 컨텍스트 매니저와 `future.result()`에 있습니다. `rfm_agent`가 리모컨 버튼을 누르면 `with` 문이 발동하면서 딱 1인용 전산실(`max_workers=1`)을 뚝딱 만듭니다. 그리고 그 안에서 논문을 열심히 수집하겠죠. 

이때 중요한 건 바로 밑에 있는 `raw = future.result()`입니다. 논문을 다 긁어오기 전까지는 다음 줄로 안 넘어가고 메인 시스템을 꽉 붙잡아둡니다. 마침내 데이터 수집이 끝나서 `raw` 값이 손에 쥐어지면, 코드가 `with` 블록 밖으로 탈출하게 되는데요. 파이썬 문법 규칙상 이 블록을 나가는 순간 **방금 전까지 일했던 1인용 전산실(스레드 풀)은 메모리에서 즉시 폭파되어 소멸**합니다.

그리고 바톤을 이어받은 다음 주자인 `vla_agent`가 다시 `call_tool`을 호출하면, 완전히 깨끗한 상태에서 새롭게 1인용 전산실을 열고 기사를 수집한 뒤 나갈 때 또 폭파시킵니다.

결과적으로 스레드 4개가 동시에 아우성치며 도는 게 아니라, **하나가 생성되어 임무를 완수하고 깔끔하게 죽고, 그다음 녀석이 바톤을 이어받아 새로 태어나는 순차적 릴레이 구조**입니다. 자원 누수나 락(Lock) 걸릴 걱정이 전혀 없는 극도로 안전하고 가벼운 인프라 구조라고 자신 있게 말씀드릴 수 있습니다."

## 14. `call_tool` 스레드 생성 - 완료(소멸) - 재생성 정밀 플로차트

에이전트 파이프라인 가동 시, 시스템 메모리 내부에서 1인 전산실(스레드 풀)이 어떻게 생겨나고 깨끗하게 소멸하는지 보여주는 타임라인 제어 흐름도입니다.

```mermaid
textplot
  ┌────────────────────────────────────────────────────────────────────────────────────────┐
  │                           메인 제어 레이어 (LangGraph 메인 스레드)                       │
  └────────────────────────────────────────────────────────────────────────────────────────┘
        │
        ├─▶ [1. RFM 노드 실행]
        │         │ 
        │         ▼ client.call_tool("search_rfm_news") 호출
        │   ┌────────────────────────────────────────────────────────┐
        │   │ ❶ 스레드방 #1 생성 (with ThreadPoolExecutor)            │
        │   │ ❷ asyncio.run() 기동 ➔ MCP 서버와 비동기 논문 수집     │
        │   │ ❸ future.result() 대기 ➔ 데이터 확보 완료                │
        │   └────────────────────────────────────────────────────────┘
        │         │ 
        │         ▼ with 블록 탈출 (스레드방 #1 및 내부 자원 완벽 파괴/소멸 ❌)
        │
        ├─▶ [2. VLA 노드 실행]
        │         │ 
        │         ▼ client.call_tool("search_vla_news") 호출
        │   ┌────────────────────────────────────────────────────────┐
        │   │ ❶ 스레드방 #2 생성 (with ThreadPoolExecutor)            │
        │   │ ❷ asyncio.run() 기동 ➔ MCP 서버와 비동기 뉴스 수집     │
        │   │ ❸ future.result() 대기 ➔ 데이터 확보 완료                │
        │   └────────────────────────────────────────────────────────┘
        │         │ 
        │         ▼ with 블록 탈출 (스레드방 #2 및 내부 자원 완벽 파괴/소멸 ❌)
        │
        ├─▶ [3. 모방학습 노드 실행]
        │         │ 
        │         ▼ client.call_tool("search_imitation_news") 호출
        │   ┌────────────────────────────────────────────────────────┐
        │   │ ❶ 스레드방 #3 생성 (with ThreadPoolExecutor)            │
        │   │ ❷ asyncio.run() 기동 ➔ MCP 서버와 비동기 자료 수집     │
        │   │ ❸ future.result() 대기 ➔ 데이터 확보 완료                │
        │   └────────────────────────────────────────────────────────┘
        │         │ 
        │         ▼ with 블록 탈출 (스레드방 #3 및 내부 자원 완벽 파괴/소멸 ❌)
        │
        └─▶ [4. 레딧 노드 실행] ➔ (동일한 ❶~❸ 스레드 생성/소멸 메커니즘 반복 후 종료)

## 15. 본 아키텍처는 어디에 쓰는 기술인가? (적용 도메인)

일반적인 공장 설비 제어(PLC, 하드웨어 인터페이스)와 달리, 본 시스템의 '순차적 스레드 생성/소멸' 구조는 **상위 데이터 플랫폼 및 정보 융합 레이어**에 최적화된 기술입니다.

### ❌ 1) 이 기술을 쓰면 안 되는 분야 (하드웨어 제어단)
* **초정밀 실시간 설비 제어:** 매 밀리초(ms) 단위의 응답성이 중요한 PLC 로직, 로봇 관절 모터 드라이버 직접 제어, 고속 비전 프레임 드랍 방지.
* **이유:** 스레드를 생성하고 소멸시키는 오버헤드(Overhead) 때문에 하드웨어 타이밍이 꼬일 수 있음. (설비단은 스레드를 미리 켜두고 재사용하는 `Thread Pool` 방식이 필수)

### 🎯 2) 이 기술이 100% 위력을 발휘하는 분야 (상위 지능 레이어)

#### ① IT 인프라와 외부 세계를 연결하는 '스마트 오피스 워크플로우'
* **예시:** 앞서 다룬 **[업무 이메일 자동 분류 및 보고서 정리 시스템]**
* **이유:** 이메일 수신, ERP 데이터 적재, 노션(Notion) 위키 업데이트 등은 ms 단위의 실시간성이 필요 없고, 수초~수분의 여유가 있습니다. 대신 외부 API 통신 중 발생하는 에러를 완벽히 격리(Fault Isolation)하는 것이 훨씬 중요하기 때문입니다.

#### ② 공정 데이터를 수집하는 '상위 통합 관제 플랫폼 (MES / 데이터 허브)'
* **예시:** 설비 가동 로그 수집, ArXiv 논문 센싱, 배달 앱 연동 등 **비정형 웹/네트워크 데이터를 긁어오는 스케줄러 애플리케이션**.
* **이유:** 간헐적으로 실행되는 배치(Batch) 작업 특성상, 평소에는 메모리를 0MB에 가깝게 완벽히 비워두었다가, 트리거가 울릴 때만 스레드를 켜서 자원을 쓰고 깔끔하게 반납하는 구조가 전체 서버 관리에 훨씬 유리합니다.

#### ③ 로봇의 '대뇌 계획(Task Planning)' 시스템 (Embodied AI 상위 레이어)
* **예시:** 로봇이 물리적으로 움직이기 전, RFM/VLA 모델을 거쳐 **"어떤 순서로 움직일지 머릿속으로 기획하고 가이드를 짜는 단계"**.
* **이유:** 기획 단계에서는 고속 연산보다 인공지능 모델과의 안전한 데이터 융합이 우선입니다. 여기서 기획된 결과 좌표(Action)가 비로소 ROS 2 인프라를 타고 내려가야 공장의 실제 로봇 관절 모터(Nachi, ABB 등)가 정밀 스레드 풀 구조로 움직이게 됩니다.

---

### 🗣️ 구조 설명

"코드를 세부적으로 뜯어보다 보면 `state["rfm_results"] = result`라는 직관적인 대입문이 나옵니다. 

이것이 바로 앞서 목청 높여 설명해 드린 **'시장 바구니의 내 칸에 물건 집어넣기'**의 실제 파이썬 코드입니다.

여기서 `state`라는 녀석은 에이전트들이 공통으로 보고 있는 거대한 '딕셔너리 메모리 장부'입니다. 처음에 프로그램을 딱 켜면 이 장부 안의 `"rfm_results"`나 `"vla_results"` 칸은 전부 비어 있습니다(`None`).

그러다가 1번 RFM 에이전트가 MCP 리모컨을 눌러서 웹사이트로부터 귀중한 최신 논문 데이터를 긁어와 `result`라는 변수에 담아냅니다. 그리고 바로 이 문제의 코드, `state["rfm_results"] = result`를 실행하는 순간, 공통 장부의 빈칸이 긁어온 논문 텍스트 데이터로 꽉 채워지게 됩니다.

이게 왜 중요하냐면, 이렇게 장부를 채워놓고 이 함수가 끝나야만, 바톤을 이어받아 잠시 후 깨어날 2번 VLA 에이전트가 장부를 열어보고 '어, 앞사람이 로봇 파운데이션 모델(RFM) 정보는 완벽하게 구해서 장부에 적어놨네? 그럼 난 내 칸인 VLA 기사만 수집해서 장부에 추가해야지' 하고 일할 수 있게 되는 겁니다.

결국 이 한 줄의 코드는 에이전트와 에이전트 사이의 **'바톤 터치'를 물리적으로 성립시키고 데이터를 안전하게 인계하는 징검다리 코드**라고 이해하시면 되겠습니다."